# Tính điểm môn và GPA trong SQLite

Dự đoán điểm môn của bộ **7, 8, 7, 9** với trọng số **10%, 20%, 20%, 50%**. Chạy từng ô bằng Shift+Enter. Ví dụ dùng dữ liệu giả lập trong bộ nhớ.

In [ ]:
from pathlib import Path
from fractions import Fraction
from contextlib import closing
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'gpa_lab.py').is_file())
sys.path.insert(0, str(root))
from gpa_lab import open_db, courses, semester_gpa, format_score, classify


## Nối bảng học kỳ và môn

`JOIN` nối khóa `courses.semester_id` với `semesters.id`. Các điểm lưu theo đơn vị 0,01 điểm nên cần chia 100 để đọc trên thang 10.

In [ ]:
with closing(open_db(':memory:', sample=True)) as db:
    rows = db.execute("""
        SELECT s.name, c.code, c.credits,
               c.process_1 / 100.0, c.process_2 / 100.0,
               c.midterm / 100.0, c.final / 100.0
        FROM semesters AS s
        JOIN courses AS c ON s.id = c.semester_id
        ORDER BY c.id
    """).fetchall()
    for row in rows:
        print(tuple(row))

## Tổng hợp theo tín chỉ

Tổng `process_1*10 + process_2*20 + midterm*20 + final*50` giữ dưới dạng số nguyên. Chia cho 10.000 để ra điểm môn. Công thức học kỳ cộng tử số đã nhân tín chỉ, rồi chia cho 10.000 nhân tổng tín chỉ. So số môn đủ điểm với tổng số môn để tránh bỏ qua một môn còn thiếu điểm.

In [ ]:
with closing(open_db(':memory:', sample=True)) as db:
    totals = db.execute("""
        WITH scores AS (
            SELECT credits,
                   process_1*10 + process_2*20 + midterm*20 + final*50 AS score
            FROM courses WHERE semester_id = ?
        )
        SELECT COUNT(*) AS subjects, COUNT(score) AS complete,
               SUM(credits) AS credits, SUM(score * credits) AS numerator
        FROM scores
    """, (1,)).fetchone()
    if totals["subjects"] and totals["subjects"] == totals["complete"]:
        gpa = Fraction(totals["numerator"], 10000 * totals["credits"])
        assert gpa == semester_gpa(courses(db, 1)) == Fraction(772, 100)
        print(format_score(gpa), classify(gpa))


## Tự kiểm chứng

Nếu môn 3 tín chỉ có điểm quá trình 1 là 0, hãy tính tác động lên GPA. Sau đó thay 0 bằng `NULL` và xem điều kiện đủ đầu điểm thay đổi.

In [ ]:
from gpa_lab import set_grades

with closing(open_db(':memory:', sample=True)) as db:
    course = courses(db, 1)[0]
    set_grades(db, course['id'], (0, 800, 700, 900))
    print(format_score(semester_gpa(courses(db, 1))))
    set_grades(db, course['id'], (None, 800, 700, 900))
    assert semester_gpa(courses(db, 1)) is None